In [2]:
using LinearAlgebra

Se implementa la función FastGivens que define los valores de $\alpha$ y $\beta$ de la matriz que elimina la segunda entrada del vector $\textbf{x}$. La función modifica a $d$. 

In [148]:
#Input: x vector 2x1, d vecor 2x1 que representa las diagonales de una matriz 2x2
#Output: α, β y el tipo de la matriz.
function FastGivens(x, d)
    if x[2] !=0 
        α = -x[1]/x[2]
        β = -α*d[2]/d[1]
        γ = -α*β
        if γ ≤ 1
            tipo = 1
            τ = d[1]
            d[1] = (1+γ)*d[2]
            d[2] = (1+γ)*τ
        else
            tipo = 2
            α = 1/α
            β = 1/β
            γ = 1/γ
            d[1] = (1+γ)*d[1]
            d[2] = (1+γ)*d[2]
        end
    else
        tipo = 2
        α = 0
        β = 0
    end
    return α, β, tipo
end


FastGivens (generic function with 1 method)

In [293]:
A = floor.(10*rand(4, 4))


4×4 Matrix{Float64}:
 9.0  9.0  6.0  2.0
 3.0  0.0  3.0  9.0
 8.0  0.0  6.0  3.0
 1.0  3.0  9.0  0.0

A continuación verificamos que se elimina la primera fila de la matriz $A$.

In [294]:
D = ones(4)
#Se copia A1 para no modificar la matriz inicial
A1 = copy(A)

for k = 4:-1:2
    d          = D[k-1:k]
    α, β, tipo = FastGivens(A1[[k-1,k],1],d)
    D[k-1:k]   = d

    #Construimos la matriz G1
    G1 = 1.0*Matrix(I, 4, 4)


    if tipo == 1
        G1[k-1:k,k-1:k] = [β 1; 1 α]
    elseif tipo == 2
        G1[k-1:k,k-1:k] = [1 α; β 1]
    end
    
    #Actualizamos A1
    A1 = G1'*A1
end

sqrt.(D).*A1

4×4 Matrix{Float64}:
 23.8239        12.911     18.4443    10.6055
  0.0          -13.2013     3.0989     6.42506
 -1.77689e-16   -0.147737   0.393964   8.42099
  0.0            3.02335    8.3142    -0.377918

Ahora se implementa FastGivensQR, la cual toma una matriz $A$ y sobreescribe en ella una matriz triangular superior $T$ y devuelve las matrices $M, D$ que satisfacen:
$$
\begin{gather} M^{T}M = D\\
M^{T}A=T\\
A=\left(MD^{-\frac{1}{2}}\right)\left(D^{-\frac{1}{2}}T\right)
\end{gather}
$$

In [288]:
#Input una matriz P, devuelve M y D. T se sobreescribe en P
function FastGivensQR(P)
    m, n = size(P)
    D = ones(m)
    
    #Para guardar la matriz M
    M = 1.0*Matrix(I, m, m)
    
    for j = 1:n
        for i = m:-1:j+1
            
            d          = D[i-1:i]
            α, β, tipo = FastGivens(P[i-1:i,j],d)
            D[i-1:i]   = d
            
            if tipo == 1
                G = [β 1; 1 α]
                P[i-1:i,j:n] = G'*P[i-1:i,j:n]
                M[:,i-1:i] = M[:,i-1:i]*G
            else
                G = [1 α; β 1]
                P[i-1:i,j:n] = G'*P[i-1:i,j:n]
                M[:,i-1:i] = M[:,i-1:i]*G
            end
            
        end
    end
    return M, D
end

FastGivensQR (generic function with 1 method)

Ejecutamos el algoritmo sobre una matriz $A$, verificamos que $A=QR$ y la ortogonalidad de $Q$

In [295]:
A  = floor.(10*rand(10,6))

10×6 Matrix{Float64}:
 1.0  2.0  8.0  0.0  7.0  0.0
 3.0  4.0  1.0  8.0  7.0  1.0
 8.0  3.0  9.0  8.0  8.0  7.0
 6.0  1.0  7.0  8.0  1.0  2.0
 2.0  2.0  7.0  8.0  4.0  1.0
 7.0  7.0  0.0  1.0  1.0  2.0
 6.0  8.0  4.0  9.0  9.0  2.0
 4.0  2.0  9.0  2.0  2.0  5.0
 7.0  8.0  2.0  6.0  9.0  4.0
 7.0  0.0  5.0  0.0  1.0  5.0

In [296]:
T = copy(A)
M, D = FastGivensQR(T)


Q = M*Diagonal(1 ./ sqrt.(D))
R = Diagonal(1 ./ sqrt.(D))*T;

In [297]:
println("La norma de M'M-D es ", opnorm(M'*M-Diagonal(D)))
println("la norma de Q'Q-I es ", opnorm(Q'*Q - UniformScaling(1)))
println("La norma de Q*R-A es ", opnorm(M'*A-T))


La norma de M'M-D es 6.660280895504657e-15
la norma de Q'Q-I es 5.968680965820366e-16
La norma de Q*R-A es 2.094552839221319e-14
